# NVIDIA NIM as Agents

This notebook will use a NVIDIA NIM with tool-calling agent capabilities in generative AI solutions. Agents can be described as AI systems that use LLMs to reason through a problem, create a plan to solve the problem, execute the plan with the help of a set of tools, and use memory to store meaningful context of the system state.

The notebook is designed to introduce tool calling, one of the agent systems' capabilities.

Tools are interfaces that accept input, execute an action, and then return a result in a structured output according to a pre-defined schema. They often encompass external API calls that the agent can use to perform tasks that go beyond the capabilities of the LLM, but do not have to be external API calls. For example, to get the current weather in San Diego, a weather tool might be used. Or to get the current score of the 49ers game, a generic web search tool or ESPN tool might be used.

You can learn more about AI Agents [here](https://www.nvidia.com/en-us/glossary/ai-agents/)


<img src="imgs/agent-components.png" alt="Agents Overview" width="600"/>


In this notebook, you will learn to use NIM in an agentic setup with the following:
### LLM
* A locally-hosted endpoint serving the Llama 3.1 8B model

### Tools
* A locally deployed NIM for Image Captioning ( Using Llama 3.2 vision instruct model)
* A API based tool to access real-time information (Tavily Search)


## Setup Environment 

In [ ]:
import getpass
import os

# del os.environ['NVIDIA_API_KEY']  ## delete key and reset
if os.environ.get("NVIDIA_API_KEY", "").startswith("nvapi-"):
    print("Valid NVIDIA_API_KEY already in environment. Delete to reset")
else:
    nvapi_key = getpass.getpass("NVAPI Key (starts with nvapi-): ")
    assert nvapi_key.startswith("nvapi-"), f"{nvapi_key[:5]}... is not a valid key"
    os.environ["NVIDIA_API_KEY"] = nvapi_key
global nvapi_key

### Deploy Local VLM

The locally deployed VLM NIM would be our first tool that the agent would call. The use of the VLM NIM will be to caption images.

Find an available port

In [ ]:
import random
import socket
import os 

def find_available_port(start=11000, end=11999):
    while True:
        # Randomly select a port between start and end range
        port = random.randint(start, end)
        
        # Try to create a socket and bind to the port
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            try:
                sock.bind(("localhost", port))
                # If binding is successful, the port is free
                return port
            except OSError:
                # If binding fails, the port is in use, continue to the next iteration
                continue

# Find and print an available port
os.environ['CONTAINER_PORT'] = str(find_available_port())
print(f"Your have been alloted the available port: {os.environ['CONTAINER_PORT']}")

In [ ]:
from os.path import expanduser
home = expanduser("~")
os.environ['LOCAL_NIM_CACHE']=f"{home}/.cache/nim"
!echo $LOCAL_NIM_CACHE

In [ ]:
!mkdir -p "$LOCAL_NIM_CACHE"
!chmod 777 "$LOCAL_NIM_CACHE"

In [ ]:
! docker run -it --rm -d \
--gpus '"device=0"' \
--name=vlm_nim \
--shm-size=16GB  \
-e NGC_API_KEY \
-v $LOCAL_NIM_CACHE:/opt/nim/.cache \
-u $(id -u) \
-p $CONTAINER_PORT:8000 \
nvcr.io/nim/meta/llama-3.2-11b-vision-instruct:1.1.1

# -e NIM_MODEL_PROFILE=tensorrt_llm-a100-bf16-tp1-throughput \
# In order to ensure, the local NIM container is completely loaded and doesn't remain in pending stage, we instantiate a wait interval
! sleep 60

In [ ]:
! docker ps -a | awk 'NR==1 || /llama/'

### Deploy Local LLM

View available models that strongly support the tool calling capabilites.

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
tool_models = [model for model in ChatNVIDIA.get_available_models() if model.supports_tools]
tool_models

The `meta/llama-3.1-8b-instruct` can be run using 1 A100/H100 80GB GPU.

In [ ]:
import random
import socket
import os 

def find_available_port(start=11000, end=11999):
    while True:
        # Randomly select a port between start and end range
        port = random.randint(start, end)
        
        # Try to create a socket and bind to the port
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            try:
                sock.bind(("localhost", port))
                # If binding is successful, the port is free
                return port
            except OSError:
                # If binding fails, the port is in use, continue to the next iteration
                continue

# Find and print an available port
os.environ['LLM_CONTAINER_PORT'] = str(find_available_port())
print(f"Your have been alloted the available port: {os.environ['LLM_CONTAINER_PORT']}")

In [ ]:
!export LOCAL_NIM_CACHE=~/.cache/nim
!mkdir -p "$LOCAL_NIM_CACHE"

!docker run -it -d --rm \
    --name=llm_nim \
    --gpus '"device=1"' \
    --shm-size=16GB \
    -e NGC_API_KEY \
    -v "$LOCAL_NIM_CACHE:/opt/nim/.cache" \
    -u $(id -u) \
    -p $LLM_CONTAINER_PORT:8000 \
    nvcr.io/nim/meta/llama-3.1-8b-instruct:1.8.3

!sleep 60

## Setup Tool Calling

In [ ]:
# === Imports ===
import base64, io
from PIL import Image
from typing import Annotated, TypedDict
from pydantic import BaseModel, Field

from langchain_core.tools import tool
from langchain_core.runnables import RunnableLambda
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langgraph.graph import END, StateGraph
from langchain_community.tools.tavily_search import TavilySearchResults

# === Image-to-base64 ===
def img2base64_string(img_path):
    print(img_path)
    image = Image.open(img_path)
    if image.width > 800 or image.height > 800:
        image.thumbnail((800, 800))
    buffered = io.BytesIO()
    image.convert("RGB").save(buffered, format="JPEG", quality=85)
    return base64.b64encode(buffered.getvalue()).decode()

### === Tool: Image Captioning ===

In [ ]:
from langchain_nvidia_ai_endpoints import register_model, Model

register_model(Model(
        id="meta/llama-3.2-11b-vision-instruct",
        model_type="vlm",
        client="ChatNVIDIA",
        endpoint=f"http://0.0.0.0:{os.environ['CONTAINER_PORT']}/v1/chat/completions",
    )
)

@tool
def ImageCaptionTool(img_path: Annotated[str, "Path to image file"]) -> str:
    """Generate a description of the image using vision LLM."""
    image_b64 = img2base64_string(img_path)
    prompt = f'What is in this image? <img src="data:image/jpeg;base64,{image_b64}" />'
    
    chat_model = ChatNVIDIA(
        model="meta/llama-3.2-11b-vision-instruct",
        max_tokens=512,
        temperature=1.00,
        top_p=1.00
    )
    
    response = chat_model.invoke(prompt)
    return response.content

Check if the image caption tool is executing correctly.

In [ ]:
img_path="./imgs/place_pic.jpg"

out=ImageCaptionTool(img_path)
out

### Expected Output

```
./imgs/place_pic.jpg
'This image depicts the Taj Mahal, a mausoleum located in Agra, India, built by Mughal Emperor Shah Jahan in memory of his wife, Mumtaz Mahal, who died during the birth of their 14th child. The Taj Mahal is an example of Mughal architecture, which combines elements of Indian, Persian, and Islamic styles. It is considered one of the most beautiful buildings in the world and is a UNESCO World Heritage Site.\n\nThe Taj Mahal is made of white marble and features intricate carvings, calligraphy, and inlays of precious stones such as jasper, jade, and turquoise. The main structure is surrounded by gardens and a reflecting pool, which creates a sense of symmetry and balance. The Taj Mahal is also decorated with flowers, birds, and other motifs, reflecting the Islamic tradition of using geometric patterns and natural forms in architecture.
```

In [ ]:
img_path="<PUT_YOUR_OWN_IMAGE_PATH>"

out=ImageCaptionTool(img_path)
out

In [ ]:
# === Location extractors ===
class ImgPathExtractor(BaseModel):
    img_path: str = Field(description="Path to the input image")

class LocationExtractor(BaseModel):
    location: str = Field(description="Name of the place described or mentioned")

# Using the 405B model for excellent tool calling capabilities
llm = ChatNVIDIA(model="meta/llama-3.1-405b-instruct", max_tokens=1024)

# OR USE the locally deployed variant.
llm = ChatNVIDIA(model="meta/llama-3.1-8b-instruct", max_tokens=1024,base_url="http://0.0.0.0:{}/v1".format(os.environ['LLM_CONTAINER_PORT']))


# === Parser functions ===
format_chain = (
    ChatPromptTemplate.from_template("Extract the image path from the input: {input}")
    | llm.with_structured_output(ImgPathExtractor)
)

def parse_image_path(state) -> dict:
    structured = format_chain.invoke({"input": state["input"]})
    return {"img_path": structured.img_path}

def extract_caption(state) -> dict:
    result = ImageCaptionTool.invoke({"img_path": state["img_path"]})
    return {"image_caption": result}

caption_to_location = (
    ChatPromptTemplate.from_template("Extract the location mentioned in this image description: {caption}")
    | llm.with_structured_output(LocationExtractor)
)

In [ ]:
def extract_location_from_caption(state) -> dict:
    print(state)
    result = caption_to_location.invoke({"caption": state["image_caption"]})
    return {"location": result.location}

text_location_chain = (
    ChatPromptTemplate.from_template("Extract the location from this user query: {input}")
    | llm.with_structured_output(LocationExtractor)
)

def extract_location_from_text(state) -> dict:
    result = text_location_chain.invoke({"input": state["input"]})
    # Add default values for image-related fields so summarizer doesn't fail
    return {
        "location": result.location,
        "img_path": "N/A",  # Default value
        "image_caption": "No image provided"  # Default value
    }

### === Tool: Web Search ===

[Tavily](https://docs.tavily.com/documentation/about) is a search engine optimized for LLMs, aimed at efficient, quick and persistent search results. Unlike other search APIs such as Serp or Google, Tavily focuses on optimizing search for AI developers and autonomous AI agents. We take care of all the burden of searching, scraping, filtering and extracting the most relevant information from online sources. All in a single API call!

In [ ]:
import getpass
import os

if "TAVILY_API_KEY" not in os.environ:
    os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key")

In [ ]:
weather_search_tool = TavilySearchResults(max_results=2)

def search_weather(state) -> dict:
    query = f"current weather in {state['location']}"
    result = weather_search_tool.invoke(query)
    return {"search_results": result}

In [ ]:
summary_prompt = ChatPromptTemplate.from_template(
    """You are an assistant summarizing visual understanding and web data.

Web search result: {search_results}

Using this, write a helpful 2-3 line summary of current weather is at that location. Also, suggest some good activities to do there. """
)

summarizer_chain = summary_prompt | llm | StrOutputParser()

def summarize_result(state) -> dict:
    summary = summarizer_chain.invoke(state)
    return {"final_response": summary}

# === Routing node ===
def route_input(state) -> dict:
    if any(ext in state["input"] for ext in [".png", ".jpg", ".jpeg"]):
        return {"has_image": True}
    else:
        return {"has_image": False}

## LangGraph Setup

[LangGraph](https://langchain-ai.github.io/langgraph/) is a low-level orchestration framework for building controllable agents. While langchain provides integrations and composable components to streamline LLM application development, the LangGraph library enables agent orchestration — offering customizable architectures, long-term memory, and human-in-the-loop to reliably handle complex tasks.

### State Setup

The first thing you do when you define a graph is define the State of the graph.
The State consists of the schema of the graph as well as reducer functions which specify how to apply updates to the state. The schema of the State will be the input schema to all Nodes and Edges in the graph, and can be either a TypedDict or a Pydantic model. All Nodes will emit updates to the State which are then applied using the specified reducer function.

In [ ]:
# === State type ===
class ImageWeatherState(TypedDict):
    input: str
    img_path: str
    image_caption: str
    location: str
    search_results: str
    final_response: str
    has_image: bool

# === Build LangGraph ===
builder = StateGraph(ImageWeatherState)

builder.add_node("route_input", RunnableLambda(route_input))
builder.add_node("parse_image_path", RunnableLambda(parse_image_path))
builder.add_node("caption_image", RunnableLambda(extract_caption))
builder.add_node("extract_location_from_caption", RunnableLambda(extract_location_from_caption))
builder.add_node("extract_location_from_text", RunnableLambda(extract_location_from_text))
builder.add_node("search_weather", RunnableLambda(search_weather))
builder.add_node("summarize", RunnableLambda(summarize_result))

builder.set_entry_point("route_input")

builder.add_conditional_edges(
    source="route_input",
    path=lambda state: "has_image" if state["has_image"] else "text_only",
    path_map={
        "has_image": "parse_image_path",
        "text_only": "extract_location_from_text"
    }
)

builder.add_edge("parse_image_path", "caption_image")
builder.add_edge("caption_image", "extract_location_from_caption")
builder.add_edge("extract_location_from_caption", "search_weather")
builder.add_edge("extract_location_from_text", "search_weather")

builder.add_edge("search_weather", "summarize")
builder.add_edge("summarize", END)

graph = builder.compile()

In [ ]:
from IPython.display import Image as img, display

try:
    display(img(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    # This requires some extra dependencies and is optional
    print(e)
    pass

## Agent Execution

#### Usecase with an image

In [ ]:
# With image
result = graph.invoke({"input": "What's the weather at the place in ./imgs/place_pic.jpg"})

# Print all steps
for k, v in result.items():
    print(f"{k.upper()}:\n{v}\n{'-'*50}")

#### Usecase without an image

In [ ]:
# Without image
result = graph.invoke({"input": "What's the weather in Tokyo?"})

# Print all steps
for k, v in result.items():
    print(f"{k.upper()}:\n{v}\n{'-'*50}")

In [ ]:
from langchain_core.runnables import RunnableConfig

from pprint import pprint

inputs = {
    "input": "What's the weather at the place in the picture ./imgs/place_pic1.jpg"
}

from langchain_core.callbacks.base import BaseCallbackHandler

class PrintStepCallback(BaseCallbackHandler):
    def on_tool_start(self, serialized, input_str, **kwargs):
        print(f"\n🔧 Tool Call: {serialized['name']} with input → {input_str}")

    def on_tool_end(self, output, **kwargs):
        print(f"✅ Tool Output: {output}")

    def on_chain_end(self, outputs, **kwargs):
        print(f"\n🔚 Final Chain Output: {outputs}")

config = RunnableConfig(callbacks=[PrintStepCallback()])
state = graph.invoke(inputs, config=config)

print("\n✅ Final Result:")
print(state['final_response'])

In [ ]:
! docker container stop vlm_nim llm_nim

## Excercise

Create and add your own tools performing custom operations to the agentic flow. 
For example: 
* Create a calculator tool (as LLMs are bad at mathematical calculations, this could be used for evaluating long mathematical expressions.)
* Create a geolocator tool that could search a place and place it on a map.
* Create a tool that creates an animated replica of a popular landmark of the place using some text2img model like stable diffusion.

This notebook showcased that you can have an NVIDIA NIM interfacing with multiple different tools and other NVIDIA NIM to create agentic workflows.

---
## Licensing

Copyright © 2025 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.